# 06 – Pipeline Testing / Evaluation
Dieses Notebook führt eine reproduzierbare Evaluation über `tests/datasets/` aus und schreibt Ergebnisse nach `tests/outputs/`.
Es vergleicht Modellantworten mit Gold-Labels und protokolliert Kennzahlen und Artefakte.


In [1]:
import csv
import json
import os
import re
import sys
from collections import Counter
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from dotenv import load_dotenv
from neo4j import GraphDatabase
from tqdm import tqdm

from haystack.components.evaluators import FaithfulnessEvaluator
from haystack.dataclasses import ChatMessage
from haystack_integrations.components.generators.ollama import OllamaChatGenerator

# -----------------------------
# Repo root + imports
# -----------------------------
# Notebook is expected to run from anywhere under the repo; we locate the repo root by finding `src/`.
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise RuntimeError(f"Repo root not found from CWD={Path.cwd().resolve()}; expected to find a 'src' directory in a parent folder.")

# Make `import src...` work reliably from notebooks/
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Load environment variables (from ROOT/.env if present)
load_dotenv(dotenv_path=ROOT / ".env")

# -----------------------------
# Config
# -----------------------------
# Supports JSONL (one JSON per line) OR a JSON array file.
TESTS_DIR = ROOT / "tests"
DATASETS_DIR = TESTS_DIR / "datasets"
OUTPUTS_DIR = TESTS_DIR / "outputs"
DATASET_PATH = DATASETS_DIR / "eval_dataset_with_gold.jsonl"
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Missing dataset file: {DATASET_PATH}. "
        f"Expected it under {DATASETS_DIR}."
    )

OLLAMA_URL = os.getenv("OLLAMA_URL", "http://ollama:11434")
GENERATOR_MODEL = os.getenv("OLLAMA_GENERATOR_MODEL") or os.getenv("OLLAMA_MODEL", "llama3.1:8b")
JUDGE_MODEL = os.getenv("OLLAMA_JUDGE_MODEL") or os.getenv("OLLAMA_GENERATOR_MODEL") or os.getenv("OLLAMA_MODEL", "llama3.1:8b")

MAX_SAMPLES: Optional[int] = None
SHOW_PROGRESS = True

SAVE_COMPONENT_OUTPUTS = True
MAX_DEBUG_DOCS = 50

# Neo4j for Graph Cypher validation (EXPLAIN)
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://neo4j:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")
_neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))


In [2]:
# -----------------------------
# Metrics + IO
# -----------------------------
def _normalize(text: str) -> str:
    text = (text or "").lower().strip()
    text = re.sub(r"[^\w\s]+", " ", text)
    return " ".join(text.split())


def exact_match(pred: str, gold: str) -> bool:
    return _normalize(pred) == _normalize(gold)


def f1_score(pred: str, gold: str) -> float:
    pred_tokens = _normalize(pred).split()
    gold_tokens = _normalize(gold).split()
    if not pred_tokens or not gold_tokens:
        return 0.0

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def load_dataset(path: Path) -> List[Dict[str, Any]]:
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        return []

    # JSON array
    if text.startswith("["):
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("JSON array expected")
        return data

    # JSONL
    rows: List[Dict[str, Any]] = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            rows.append(json.loads(line))
    return rows


def save_json(path: Path, payload: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def write_csv(path: Path, rows: List[Dict[str, Any]], fieldnames: List[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: row.get(k, "") for k in fieldnames})

In [3]:
# -----------------------------
# Graph Cypher checks
# -----------------------------
_FORBIDDEN = ["create", "merge", "delete", "set", "call", "load csv", "apoc", "drop", "remove"]


def cypher_policy_check(cypher: Optional[str]) -> Tuple[bool, Optional[str]]:
    if not cypher:
        return False, "empty cypher"
    s = cypher.strip()
    if "```" in s:
        return False, "code fence detected"
    if s.lower().startswith("cypher"):
        s = s.split(":", 1)[-1].strip()
    low = s.lower()
    for kw in _FORBIDDEN:
        if kw in low:
            return False, f"forbidden keyword: {kw}"
    if ";" in s.strip("; "):
        return False, "multiple statements / semicolon detected"
    if "return" not in low:
        return False, "missing RETURN"
    if not (low.startswith("match") or low.startswith("optional match")):
        return False, "query must start with MATCH"
    return True, None


def cypher_explain_ok(cypher: Optional[str]) -> Tuple[bool, Optional[str]]:
    ok, reason = cypher_policy_check(cypher)
    if not ok:
        return False, reason
    q = (cypher or "").strip()
    try:
        with _neo4j_driver.session() as session:
            session.run("EXPLAIN " + q).consume()
        return True, None
    except Exception as exc:
        return False, str(exc)


In [4]:
# -----------------------------
# Faithfulness judge
# -----------------------------
def _extract_json_object(text: Optional[str]) -> Optional[str]:
    if not text:
        return None
    text = text.strip()

    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            return json.dumps(parsed, ensure_ascii=False)
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None

    candidate = text[start : end + 1]
    try:
        parsed = json.loads(candidate)
        if isinstance(parsed, dict):
            return json.dumps(parsed, ensure_ascii=False)
    except Exception:
        return None

    return None


class JSONCoercingChatGenerator:
    def __init__(self, base: OllamaChatGenerator):
        self._base = base

    def run(self, messages: List[ChatMessage]):
        res = self._base.run(messages=messages)
        replies = res.get("replies", [])
        if replies and replies[0].text:
            coerced = _extract_json_object(replies[0].text)
            if coerced is not None:
                replies[0] = ChatMessage.from_assistant(coerced)
        return {"replies": replies}


faithfulness_response_schema = {
    "type": "object",
    "properties": {
        "statements": {"type": "array", "items": {"type": "string"}},
        "statement_scores": {"type": "array", "items": {"type": "integer", "enum": [0, 1]}},
    },
    "required": ["statements", "statement_scores"],
}

judge_llm = OllamaChatGenerator(
    model=JUDGE_MODEL,
    url=OLLAMA_URL,
    generation_kwargs={"temperature": 0.0, "num_predict": 256},
    response_format=faithfulness_response_schema,
)
faithfulness = FaithfulnessEvaluator(
    chat_generator=JSONCoercingChatGenerator(judge_llm),
    progress_bar=False,
    raise_on_failure=True,
)


def faithfulness_single(*, question: str, context: str, predicted_answer: str, retries: int = 2) -> Tuple[Optional[float], Optional[dict], Optional[str]]:
    last_err: Optional[Exception] = None
    for _ in range(retries + 1):
        try:
            res = faithfulness.run(
                questions=[question],
                contexts=[[context]],
                predicted_answers=[predicted_answer],
            )
            score = res.get("individual_scores", [None])[0]
            details = res.get("results", [None])[0]
            return score, details, None
        except Exception as exc:
            last_err = exc
    return None, None, str(last_err) if last_err else "unknown error"


PromptBuilder has 3 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


In [5]:
# -----------------------------
# Pipelines
# -----------------------------
from src.graph.graph_pipeline import build_pipeline as build_graph_pipeline
from src.base.base_pipeline import create_rag_pipeline as build_base_pipeline, run_rag_pipeline

graph_pipe = build_graph_pipeline()
base_pipe = build_base_pipeline(ollama_model=GENERATOR_MODEL, ollama_url=OLLAMA_URL)


In [8]:
# -----------------------------
# Run helpers
# -----------------------------
def run_graph_with_debug(pipe, question: str) -> Tuple[str, str, Optional[str], Dict[str, Any]]:
    include = ["query_processor", "retriever", "organizer", "generator"]
    result = pipe.run(
        {
            "query_processor": {"query": question},
            "generator": {"query": question},
        },
        include_outputs_from=include,
    )

    answer = (result.get("generator", {}) or {}).get("answer", "")
    analysis_json = (result.get("organizer", {}) or {}).get("analysis_json")
    if isinstance(analysis_json, dict):
        context = json.dumps(analysis_json, ensure_ascii=False)
    else:
        context = str(analysis_json or "")
    cypher = (result.get("query_processor", {}) or {}).get("cypher")

    debug: Dict[str, Any] = {
        "query_processor": {"cypher": cypher},
        "organizer": {"analysis_json": analysis_json},
        "generator": {"answer": answer, "model": GENERATOR_MODEL},
    }

    docs = (result.get("retriever", {}) or {}).get("documents")
    if isinstance(docs, list):
        debug["retriever"] = {
            "documents_total": len(docs),
            "documents": [
                {"content": getattr(d, "content", None), "meta": getattr(d, "meta", None)}
                for d in docs[:MAX_DEBUG_DOCS]
            ],
        }
    else:
        debug["retriever"] = {"documents_total": None, "documents": None}

    return answer, context, cypher, debug


def _extract_base_answer(result: Dict[str, Any]) -> str:
    llm = result.get("llm")
    if isinstance(llm, dict):
        replies = llm.get("replies")
        if isinstance(replies, list) and replies:
            return str(replies[0])
        if replies is not None:
            return str(replies)
    return str(result.get("llm", ""))


def _extract_base_context(result: Dict[str, Any], max_docs: int = 50) -> str:
    docs = None
    retr = result.get("retriever")
    if isinstance(retr, dict):
        docs = retr.get("documents")
    if not docs:
        pb = result.get("prompt_builder")
        if isinstance(pb, dict):
            docs = pb.get("documents")
    if isinstance(docs, list):
        parts = []
        for d in docs[:max_docs]:
            content = getattr(d, "content", None)
            if content:
                parts.append(str(content))
        return "\n".join(parts)
    return ""


def run_base_with_debug(pipe, question: str) -> Tuple[str, str, Dict[str, Any]]:
    result = run_rag_pipeline(pipe, question, debug=False, preview_docs=0)
    answer = _extract_base_answer(result)
    context = _extract_base_context(result)

    debug: Dict[str, Any] = {
        "retriever": {"documents_total": None, "documents": None},
        "prompt_builder": {},
        "llm": {"answer": answer, "model": GENERATOR_MODEL},
    }

    docs = None
    retr = result.get("retriever")
    if isinstance(retr, dict):
        docs = retr.get("documents")
    if isinstance(docs, list):
        debug["retriever"]["documents_total"] = len(docs)
        debug["retriever"]["documents"] = [
            {"content": getattr(d, "content", None), "meta": getattr(d, "meta", None)}
            for d in docs[:MAX_DEBUG_DOCS]
        ]

    return answer, context, debug

In [ ]:
# -----------------------------
# Evaluation (both pipelines + comparison)
# -----------------------------
def evaluate_both(*, graph_pipe, base_pipe, dataset_path: Path, max_samples: Optional[int] = None, show_progress: bool = True) -> Dict[str, Any]:
    data = load_dataset(dataset_path)
    if max_samples is not None:
        data = data[:max_samples]

    run_id = datetime.utcnow().strftime("%Y%m%dT%H%M%S")
    logs_dir = OUTPUTS_DIR / "compare" / "logs" / run_id
    logs_dir.mkdir(parents=True, exist_ok=True)

    rows: List[Dict[str, Any]] = []
    iterator = tqdm(data, disable=not show_progress, desc="Evaluating (both)")

    for idx, row in enumerate(iterator):
        q = str(row.get("question", row.get("query", "")))
        gold = str(row.get("gold_answer", ""))

        # --- Graph ---
        g_pred, g_ctx, g_cypher, g_dbg = "", "", None, {}
        g_err: Optional[str] = None
        try:
            g_pred, g_ctx, g_cypher, g_dbg = run_graph_with_debug(graph_pipe, q)
        except Exception as exc:
            g_err = str(exc)
            g_pred, g_ctx = f"[ERROR] {exc}", ""
            g_dbg = {"error": g_err}

        g_cy_policy_ok, g_cy_policy_err = cypher_policy_check(g_cypher)
        g_cy_explain_ok, g_cy_explain_err = cypher_explain_ok(g_cypher)
        g_cy_err = g_cy_policy_err if not g_cy_policy_ok else g_cy_explain_err

        g_em = exact_match(g_pred, gold)
        g_f1 = f1_score(g_pred, gold)
        g_faith, g_faith_details, g_faith_err = faithfulness_single(question=q, context=g_ctx, predicted_answer=g_pred, retries=2)

        # --- Base ---
        b_pred, b_ctx, b_dbg = "", "", {}
        b_err: Optional[str] = None
        try:
            b_pred, b_ctx, b_dbg = run_base_with_debug(base_pipe, q)
        except Exception as exc:
            b_err = str(exc)
            b_pred, b_ctx = f"[ERROR] {exc}", ""
            b_dbg = {"error": b_err}

        b_em = exact_match(b_pred, gold)
        b_f1 = f1_score(b_pred, gold)
        b_faith, b_faith_details, b_faith_err = faithfulness_single(question=q, context=b_ctx, predicted_answer=b_pred, retries=2)

        out_row: Dict[str, Any] = {
            "id": row.get("id"),
            "category": row.get("category"),
            "question": q,
            "gold_answer": gold,

            "graph_prediction": g_pred,
            "graph_exact_match": g_em,
            "graph_f1": g_f1,
            "graph_faithfulness": g_faith,
            "graph_faithfulness_error": g_faith_err,
            "graph_cypher": g_cypher,
            "graph_cypher_policy_ok": g_cy_policy_ok,
            "graph_cypher_explain_ok": g_cy_explain_ok,
            "graph_cypher_error": g_cy_err,
            "graph_error": g_err,

            "base_prediction": b_pred,
            "base_exact_match": b_em,
            "base_f1": b_f1,
            "base_faithfulness": b_faith,
            "base_faithfulness_error": b_faith_err,
            "base_error": b_err,

            "generator_model": GENERATOR_MODEL,
            "judge_model": JUDGE_MODEL,
        }

        log_path = logs_dir / f"{idx:04d}_{row.get('id', 'sample')}.json"
        out_row["log_path"] = str(log_path)

        if SAVE_COMPONENT_OUTPUTS:
            out_row["component_outputs"] = {"graph": g_dbg, "base": b_dbg, "gold_row": row}
            save_json(log_path, out_row)

        rows.append(out_row)

    def _avg(values: List[Optional[float]]) -> float:
        vals = [v for v in values if isinstance(v, (int, float))]
        return float(sum(vals) / len(vals)) if vals else 0.0

    summary = {
        "run_id": run_id,
        "samples": len(rows),
        "dataset": str(dataset_path),
        "ollama_url": OLLAMA_URL,
        "generator_model": GENERATOR_MODEL,
        "judge_model": JUDGE_MODEL,
        "logs_dir": str(logs_dir),
        "graph_exact_match": round(sum(1 for r in rows if r.get('graph_exact_match')) / len(rows), 4) if rows else 0.0,
        "graph_avg_f1": round(sum(float(r.get('graph_f1', 0.0)) for r in rows) / len(rows), 4) if rows else 0.0,
        "graph_faithfulness": round(_avg([r.get('graph_faithfulness') for r in rows]), 4),
        "graph_cypher_explain_ok_rate": round(sum(1 for r in rows if r.get('graph_cypher_explain_ok') is True) / len(rows), 4) if rows else 0.0,
        "base_exact_match": round(sum(1 for r in rows if r.get('base_exact_match')) / len(rows), 4) if rows else 0.0,
        "base_avg_f1": round(sum(float(r.get('base_f1', 0.0)) for r in rows) / len(rows), 4) if rows else 0.0,
        "base_faithfulness": round(_avg([r.get('base_faithfulness') for r in rows]), 4),
    }

    results_path = OUTPUTS_DIR / "compare" / f"results_{run_id}.json"
    save_json(results_path, {"summary": summary, "results": rows})

    fieldnames = [
        'id','category','question','gold_answer',
        'graph_prediction','graph_exact_match','graph_f1','graph_faithfulness',
        'graph_cypher','graph_cypher_policy_ok','graph_cypher_explain_ok','graph_cypher_error',
        'base_prediction','base_exact_match','base_f1','base_faithfulness',
        'log_path','generator_model','judge_model',
    ]
    csv_path = OUTPUTS_DIR / "compare" / f"summary_{run_id}.csv"
    write_csv(csv_path, rows, fieldnames)

    return {"summary": summary, "json_path": str(results_path), "csv_path": str(csv_path), "logs_dir": str(logs_dir)}


evaluation = evaluate_both(
    graph_pipe=graph_pipe,
    base_pipe=base_pipe,
    dataset_path=DATASET_PATH,
    max_samples=MAX_SAMPLES,
    show_progress=SHOW_PROGRESS,
 )

print('Summary:')
print(evaluation['summary'])
print('Saved JSON:', evaluation['json_path'])
print('Saved CSV:', evaluation['csv_path'])
print('Logs dir:', evaluation['logs_dir'])


Evaluating (both):   4%|▍         | 4/100 [01:24<34:20, 21.46s/it]